# FinAgent-LoRA: fine-tune a small model into a reliable financial agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ajinkyarodge66/finagent-lora/blob/main/notebooks/train_and_eval.ipynb)

This notebook:
1. Builds verified tool-calling trajectories from [FinAgent-Bench](https://github.com/Ajinkyarodge66/finagent-bench) (training seeds never overlap the test seed).
2. Evaluates the **base** model on FinAgent-Bench.
3. LoRA fine-tunes it (assistant-only loss) and evaluates again.
4. Optional ablations: train **without injected examples**, and train **without multi-hop tasks**, to see where robustness and generalisation come from.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**. Expect about 45-60 minutes with ablations, about 20 without.

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
%pip install -q "finagent-lora @ git+https://github.com/Ajinkyarodge66/finagent-lora"

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # any chat model with a tool-calling template
RUN_ABLATIONS = True                    # set False for a quicker run (base + main LoRA only)
TAG = "qwen2.5-1.5b"

## 1. Training data

In [ ]:
import subprocess, sys
def sh(*args): print(subprocess.run([sys.executable, "-m", *args], check=True, capture_output=True, text=True).stdout)
sh("finagent_lora.data", "--out", "data/train.jsonl")
if RUN_ABLATIONS:
    sh("finagent_lora.data", "--out", "data/train_noinj.jsonl", "--injection-rate", "0")
    sh("finagent_lora.data", "--out", "data/train_nomh.jsonl", "--exclude", "multi_hop")

### Sanity check: what does the model actually learn from?
Only the assistant's actions should be trained on. Documents and questions must be masked out.

In [ ]:
import json
from transformers import AutoTokenizer
from finagent_lora.train import assistant_token_mask, hf_tools, IGNORE
tok = AutoTokenizer.from_pretrained(MODEL)
ex = json.loads(open("data/train.jsonl").readline())["messages"]
ids, labels = assistant_token_mask(tok, ex, hf_tools())
trained = tok.decode([t for t, l in zip(ids, labels) if l != IGNORE])
print(f"{sum(l != IGNORE for l in labels)} of {len(ids)} tokens trained on:\n")
print(trained)
assert "ANNUAL REPORT" not in trained and "final_answer" in trained

## 2. Base model on FinAgent-Bench

In [ ]:
import gc
from finagent_lora.evaluate import load, run
from finagent_lora.train import train

def evaluate(name, adapter=None):
    model, tok = load(MODEL, adapter)
    r = run(model, tok, name)
    del model; gc.collect(); torch.cuda.empty_cache()
    return r

evaluate(f"{TAG}-base")

## 3. LoRA fine-tune (all task types, 30% injected documents)

In [ ]:
meta = train(MODEL, "data/train.jsonl", "runs/lora")
gc.collect(); torch.cuda.empty_cache()
evaluate(f"{TAG}-lora", "runs/lora")

## 4. Ablations
- **no-injection**: same data but no injected documents. Does robustness come from the training data, or from learning the task format?
- **no-multi-hop**: multi-hop tasks removed from training. Can the model generalise to reading and combining two filings?

In [ ]:
if RUN_ABLATIONS:
    train(MODEL, "data/train_noinj.jsonl", "runs/lora_noinj"); gc.collect(); torch.cuda.empty_cache()
    evaluate(f"{TAG}-lora-noinj", "runs/lora_noinj")
    train(MODEL, "data/train_nomh.jsonl", "runs/lora_nomh"); gc.collect(); torch.cuda.empty_cache()
    evaluate(f"{TAG}-lora-nomh", "runs/lora_nomh")

## 5. Summary

In [ ]:
import glob, os
from finagent_bench.evaluate import to_markdown
results = [json.load(open(p)) for p in sorted(glob.glob("results/*.json"))]
print(to_markdown(results))
for p in sorted(glob.glob("runs/*/train_meta.json")):
    m = json.load(open(p)); print(f"{p}: {m['examples']} examples, {m['minutes']} min, loss {m['loss_curve'][0][1]:.3f} -> {m['loss_curve'][-1][1]:.3f}")

In [ ]:
import shutil
os.makedirs("export/runs", exist_ok=True)
for p in glob.glob("runs/*/train_meta.json"):
    shutil.copy(p, f"export/runs/{p.split('/')[1]}_train_meta.json")
shutil.copytree("results", "export/results", dirs_exist_ok=True)
shutil.make_archive("finagent_lora_results", "zip", "export")
from google.colab import files
files.download("finagent_lora_results.zip")

**Done.** `finagent_lora_results.zip` contains the results and failure reports for every model plus the training logs. Add them to the repo's `results/` folder.